# Exploratory Data Analysis — Stroke Dataset

Notebook này khám phá bộ dữ liệu dự đoán nguy cơ đột quỵ. Mục tiêu của EDA là hiểu cấu trúc dữ liệu, phát hiện dữ liệu thiếu, kiểm tra mất cân bằng và tìm các biến có liên hệ với `stroke`.

## 1. Import thư viện và thiết lập đường dẫn

Dữ liệu được đặt trong `data/raw`, còn các bảng và biểu đồ EDA được lưu trong `outputs/eda`. Dùng backend `Agg` để notebook vẫn xuất được ảnh trong môi trường không có giao diện đồ họa.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'healthcare-dataset-stroke-data.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'eda'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Data path:', DATA_PATH)

## 2. Đọc dữ liệu và kiểm tra tổng quan

`N/A` được chuyển thành giá trị thiếu chuẩn của pandas để có thể thống kê và xử lý chính xác.

In [ ]:
df = pd.read_csv(DATA_PATH, na_values=['N/A'])
display(df.head())
print(f'Kích thước: {df.shape[0]:,} dòng x {df.shape[1]} cột')
display(df.info())

## 3. Chất lượng dữ liệu

Kiểm tra kiểu dữ liệu, giá trị thiếu, dòng trùng và mã định danh. Cột `id` chỉ là mã hồ sơ nên không đưa vào phân tích dự báo.

In [ ]:
quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2),
    'n_unique': df.nunique(dropna=False),
})
display(quality)
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate ids:', df['id'].duplicated().sum())
quality.to_csv(OUTPUT_DIR / 'data_quality.csv', encoding='utf-8-sig')

## 4. Phân tích biến mục tiêu

`stroke = 1` là có đột quỵ. Tỷ lệ thấp cho thấy dữ liệu mất cân bằng; khi xây dựng mô hình cần ưu tiên các chỉ số như recall, F1 và PR-AUC thay vì chỉ dùng accuracy.

In [ ]:
target_summary = df['stroke'].value_counts().rename_axis('stroke').to_frame('count')
target_summary['percent'] = (target_summary['count'] / len(df) * 100).round(2)
display(target_summary)

plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='stroke')
ax.set(title='Stroke target distribution', xlabel='Stroke', ylabel='Count')
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_target_distribution.png', dpi=150)
plt.show()
plt.close()

## 5. Phân tích các biến số

So sánh tuổi, đường huyết trung bình và BMI giữa hai nhóm. Boxplot giúp quan sát trung vị, độ phân tán và các giá trị ngoại lệ.

In [ ]:
numeric_cols = ['age', 'avg_glucose_level', 'bmi']
display(df[numeric_cols + ['stroke']].groupby('stroke').describe().T)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, numeric_cols):
    sns.histplot(data=df, x=col, hue='stroke', kde=True, stat='density',
                 common_norm=False, element='step', ax=ax)
    ax.set_title(f'{col} by stroke')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_numeric_distributions.png', dpi=150)
plt.show()
plt.close()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(data=df, x='stroke', y=col, ax=ax)
    ax.set_title(f'{col} vs stroke')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_boxplots_by_stroke.png', dpi=150)
plt.show()
plt.close()

## 6. Phân tích biến phân loại

Tính số lượng quan sát và tỷ lệ đột quỵ trong từng nhóm. Các nhóm có rất ít quan sát cần được diễn giải thận trọng.

In [ ]:
categorical_cols = df.drop(columns=['id', 'stroke']).select_dtypes(include='object').columns
category_rates = []
for col in categorical_cols:
    rates = df.groupby(col, dropna=False)['stroke'].agg(count='size', strokes='sum', stroke_rate='mean').reset_index()
    rates['stroke_rate_percent'] = (rates['stroke_rate'] * 100).round(2)
    rates['feature'] = col
    category_rates.append(rates)
display(pd.concat(category_rates, ignore_index=True))

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, col in zip(axes.flat, categorical_cols):
    rates = df.groupby(col, dropna=False)['stroke'].mean().mul(100).sort_values(ascending=False)
    sns.barplot(x=rates.values, y=rates.index, ax=ax)
    ax.set(title=f'Stroke rate by {col}', xlabel='Stroke rate (%)', ylabel='')
axes.flat[-1].axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_categorical_stroke_rates.png', dpi=150)
plt.show()
plt.close()

## 7. Tương quan và kết luận EDA

Tương quan chỉ mô tả mối liên hệ tuyến tính, không chứng minh quan hệ nhân quả. Kết quả EDA cần được dùng để định hướng tiền xử lý và xây dựng mô hình tiếp theo.

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.drop(columns=['id']).select_dtypes(exclude='object').corr(),
            annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Numeric feature correlation')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_correlation_heatmap.png', dpi=150)
plt.show()
plt.close()

print('Kết luận:')
print('- Dữ liệu mất cân bằng mạnh, stroke chiếm khoảng 4.87%.')
print('- bmi có 201 giá trị thiếu, cần impute trước khi huấn luyện mô hình.')
print('- age, avg_glucose_level, hypertension và heart_disease đáng được ưu tiên kiểm tra ở bước modeling.')

## 8. Imbalance và lựa chọn metric

Lớp `stroke = 1` chỉ chiếm khoảng 4,87%, trong khi lớp không đột quỵ chiếm 95,13%. Nếu mô hình luôn dự đoán không đột quỵ thì Accuracy đã đạt 95,13%, nhưng Recall của lớp đột quỵ bằng 0 — mô hình bỏ sót toàn bộ ca bệnh. Vì vậy không dùng Accuracy làm metric chính.

- **Recall**: đo khả năng phát hiện đúng ca đột quỵ; quan trọng khi chi phí bỏ sót ca bệnh cao.
- **F1-score**: cân bằng giữa Precision và Recall tại một ngưỡng phân loại cụ thể.
- **AUC-PR**: đánh giá chất lượng xếp hạng xác suất trên lớp dương trong dữ liệu mất cân bằng, ít bị chi phối bởi số lượng âm tính.

**Metric chính được chọn: AUC-PR (Average Precision).** Dùng Recall và F1-score làm metric phụ; khi triển khai cần chọn threshold dựa trên mức Recall tối thiểu chấp nhận được.

In [ ]:
from sklearn.metrics import average_precision_score, f1_score, recall_score

positive_rate = df['stroke'].mean()
negative_rate = 1 - positive_rate
y_true = df['stroke']
y_pred_baseline = pd.Series(0, index=df.index)
y_score_baseline = pd.Series(positive_rate, index=df.index)

baseline_metrics = pd.Series({
    'positive_rate_percent': positive_rate * 100,
    'negative_rate_percent': negative_rate * 100,
    'majority_accuracy_percent': negative_rate * 100,
    'recall': recall_score(y_true, y_pred_baseline, zero_division=0),
    'f1': f1_score(y_true, y_pred_baseline, zero_division=0),
    'auc_pr_average_precision': average_precision_score(y_true, y_score_baseline),
})
display(baseline_metrics.to_frame('value').round(4))
baseline_metrics.to_frame('value').to_csv(OUTPUT_DIR / 'imbalance_baseline_metrics.csv', encoding='utf-8-sig')
print('Kết luận: AUC-PR là metric chính; Recall và F1 là metric phụ.')

## 9. So sánh chiến lược xử lý imbalance

Các chiến lược được đánh giá trên cùng mô hình Logistic Regression và cùng cách chia dữ liệu có stratify. Tiền xử lý và oversampling chỉ được fit trên tập train để tránh data leakage.

1. **Class weight**: tăng trọng số cho lớp đột quỵ trong hàm loss; đơn giản, không tạo dữ liệu giả.
2. **SMOTE**: tạo mẫu dương tổng hợp dựa trên các láng giềng gần nhất; có thể cải thiện Recall nhưng có nguy cơ tạo mẫu không thực tế.
3. **ADASYN**: tạo nhiều mẫu hơn ở vùng khó học; linh hoạt nhưng có thể khuếch đại nhiễu/outlier.
4. **Threshold tuning**: giữ nguyên xác suất dự đoán nhưng thay đổi ngưỡng từ 0,5; phù hợp khi cần ưu tiên Recall hoặc F1 theo mục tiêu nghiệp vụ.

AUC-PR được dùng để chọn chiến lược tổng thể; Recall và F1 được báo cáo ở threshold triển khai.

In [ ]:
# Cần cài imbalanced-learn nếu môi trường chưa có: pip install imbalanced-learn
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from imblearn.over_sampling import ADASYN, SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

model_df = df.drop(columns=['id']).copy()
X = model_df.drop(columns=['stroke'])
y = model_df['stroke']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

numeric_features = ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease']
categorical_features = [c for c in X.columns if c not in numeric_features]
preprocessor = ColumnTransformer([
    ('num', SklearnPipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_features),
    ('cat', SklearnPipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
])

def evaluate(name, estimator, threshold=0.5):
    estimator.fit(X_fit, y_fit)
    scores = estimator.predict_proba(X_test)[:, 1]
    predictions = (scores >= threshold).astype(int)
    return {'strategy': name, 'threshold': threshold,
            'auc_pr': average_precision_score(y_test, scores),
            'recall': recall_score(y_test, predictions, zero_division=0),
            'f1': f1_score(y_test, predictions, zero_division=0)}

def make_pipeline(class_weight=None, sampler=None):
    steps = [('preprocess', preprocessor)]
    if sampler is not None:
        steps.append(('sampler', sampler))
    steps.append(('model', LogisticRegression(max_iter=2000, class_weight=class_weight, random_state=42)))
    return ImbPipeline(steps)

results = []
results.append(evaluate('class_weight=balanced', make_pipeline(class_weight='balanced')))
results.append(evaluate('SMOTE', make_pipeline(sampler=SMOTE(random_state=42))))
results.append(evaluate('ADASYN', make_pipeline(sampler=ADASYN(random_state=42))))

# Threshold tuning: chọn threshold tối ưu F1 trên validation, rồi đánh giá một lần trên test.
threshold_model = make_pipeline(class_weight='balanced')
threshold_model.fit(X_fit, y_fit)
val_scores = threshold_model.predict_proba(X_val)[:, 1]
thresholds = np.arange(0.05, 0.96, 0.01)
best_threshold = max(thresholds, key=lambda t: f1_score(y_val, val_scores >= t, zero_division=0))
results.append(evaluate('class_weight + threshold tuning', threshold_model, threshold=best_threshold))

comparison = pd.DataFrame(results).sort_values('auc_pr', ascending=False)
display(comparison.round(4))
comparison.to_csv(OUTPUT_DIR / 'imbalance_strategy_comparison.csv', index=False, encoding='utf-8-sig')